# All Models Demo: LSTM, Random Forest, and XGBoost/AdaBoost

This notebook demonstrates how to use LSTM, Random Forest, and XGBoost/AdaBoost models with completely unrelated mock data (sales/time-series data). This focuses on model usage patterns rather than domain-specific knowledge.

## Table of Contents

1. [LSTM Demo](#section1)
2. [Random Forest Demo](#section2)
3. [XGBoost/AdaBoost Demo](#section3)
4. [Common Patterns](#section4)


## Section 1: LSTM Demo {#section1}

We'll use synthetic daily sales data to demonstrate LSTM for time-series forecasting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Create synthetic daily sales data (2 years)
np.random.seed(42)
n_days = 730  # 2 years
dates = pd.date_range(start='2022-01-01', periods=n_days, freq='D')

# Generate synthetic sales with trend, seasonality, and noise
trend = np.linspace(1000, 1500, n_days)
seasonality = 200 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)  # Annual seasonality
weekly_pattern = 50 * np.sin(2 * np.pi * np.arange(n_days) / 7)  # Weekly pattern
noise = np.random.normal(0, 50, n_days)
sales = trend + seasonality + weekly_pattern + noise

# Create DataFrame
df_sales = pd.DataFrame({
    'Date': dates,
    'Sales': sales
})

print("Sales Data Overview:")
print(df_sales.head())
print(f"\nShape: {df_sales.shape}")
print(f"\nDate range: {df_sales['Date'].min()} to {df_sales['Date'].max()}")
print(f"\nSales statistics:")
print(df_sales['Sales'].describe())


In [ ]:
# Visualize the sales data
plt.figure(figsize=(14, 6))
plt.plot(df_sales['Date'], df_sales['Sales'], linewidth=0.5)
plt.title('Daily Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Prepare Sequences for LSTM

LSTM requires sequences of data as input. We'll create sequences where each sample uses the previous N days to predict the next day.


In [ ]:
# Scale the data (LSTM works better with scaled data)
scaler = MinMaxScaler()
sales_scaled = scaler.fit_transform(df_sales[['Sales']])

# Parameters for sequence creation
lookback = 30  # Use 30 days to predict next day
forecast_horizon = 1  # Predict 1 day ahead

# Create sequences
X, y = [], []
for i in range(lookback, len(sales_scaled) - forecast_horizon + 1):
    X.append(sales_scaled[i-lookback:i, 0])
    y.append(sales_scaled[i+forecast_horizon-1, 0])

X = np.array(X)
y = np.array(y)

# Reshape for LSTM: (samples, time_steps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

print(f"Sequence shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nEach sequence uses {lookback} days to predict {forecast_horizon} day(s) ahead")


In [ ]:
# Split data (maintain temporal order - don't shuffle!)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Further split training data for validation
val_split_idx = int(len(X_train) * 0.8)
X_val, X_train = X_train[val_split_idx:], X_train[:val_split_idx]
y_val, y_train = y_train[val_split_idx:], y_train[:val_split_idx]

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")


### Build LSTM Model Architecture


In [ ]:
# Build LSTM model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(lookback, 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

# Compile model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# Display model architecture
model.summary()


In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    validation_data=(X_val, y_val),
    verbose=1,
    shuffle=False  # Don't shuffle time-series data!
)


In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Model MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Make predictions
y_train_pred = model.predict(X_train, verbose=0)
y_val_pred = model.predict(X_val, verbose=0)
y_test_pred = model.predict(X_test, verbose=0)

# Inverse transform to original scale
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()
y_train_pred_actual = scaler.inverse_transform(y_train_pred).flatten()
y_val_actual = scaler.inverse_transform(y_val.reshape(-1, 1)).flatten()
y_val_pred_actual = scaler.inverse_transform(y_val_pred).flatten()
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_test_pred_actual = scaler.inverse_transform(y_test_pred).flatten()

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual))
val_rmse = np.sqrt(mean_squared_error(y_val_actual, y_val_pred_actual))
test_rmse = np.sqrt(mean_squared_error(y_test_actual, y_test_pred_actual))

train_mae = mean_absolute_error(y_train_actual, y_train_pred_actual)
val_mae = mean_absolute_error(y_val_actual, y_val_pred_actual)
test_mae = mean_absolute_error(y_test_actual, y_test_pred_actual)

train_r2 = r2_score(y_train_actual, y_train_pred_actual)
val_r2 = r2_score(y_val_actual, y_val_pred_actual)
test_r2 = r2_score(y_test_actual, y_test_pred_actual)

print("LSTM Model Performance:")
print(f"\nTraining Set:")
print(f"  RMSE: {train_rmse:.2f}")
print(f"  MAE: {train_mae:.2f}")
print(f"  R²: {train_r2:.4f}")

print(f"\nValidation Set:")
print(f"  RMSE: {val_rmse:.2f}")
print(f"  MAE: {val_mae:.2f}")
print(f"  R²: {val_r2:.4f}")

print(f"\nTest Set:")
print(f"  RMSE: {test_rmse:.2f}")
print(f"  MAE: {test_mae:.2f}")
print(f"  R²: {test_r2:.4f}")


In [ ]:
# Visualize predictions
plt.figure(figsize=(14, 8))

# Test set predictions
test_dates = df_sales['Date'].iloc[split_idx+lookback+1:split_idx+lookback+1+len(y_test_actual)]

plt.subplot(2, 1, 1)
plt.plot(test_dates, y_test_actual, label='Actual', alpha=0.7, linewidth=1)
plt.plot(test_dates, y_test_pred_actual, label='Predicted', alpha=0.7, linewidth=1)
plt.title('LSTM Test Set Predictions')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True, alpha=0.3)

# Scatter plot
plt.subplot(2, 1, 2)
plt.scatter(y_test_actual, y_test_pred_actual, alpha=0.5)
plt.plot([y_test_actual.min(), y_test_actual.max()], 
         [y_test_actual.min(), y_test_actual.max()], 'r--', lw=2)
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted (Test Set)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Save the model
model.save('lstm_sales_model.h5')
print("Model saved as 'lstm_sales_model.h5'")

# Example: Load the model
# loaded_model = keras.models.load_model('lstm_sales_model.h5')


## Section 2: Random Forest Demo {#section2}

We'll use synthetic customer data to demonstrate Random Forest for regression and classification.


In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Create synthetic customer data
np.random.seed(42)
n_samples = 1000

data = {
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.normal(50000, 20000, n_samples),
    'years_member': np.random.randint(0, 10, n_samples),
    'purchases_last_year': np.random.randint(0, 50, n_samples),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n_samples),
    'subscription_type': np.random.choice(['Basic', 'Premium', 'Gold'], n_samples)
}

df_customers = pd.DataFrame(data)

# Create regression target: purchase amount
df_customers['purchase_amount'] = (
    100 * df_customers['income'] / 10000 +
    50 * df_customers['years_member'] +
    20 * df_customers['purchases_last_year'] +
    np.random.normal(0, 100, n_samples)
)

# Create classification target: high value customer (1) or not (0)
df_customers['high_value'] = (df_customers['purchase_amount'] > df_customers['purchase_amount'].median()).astype(int)

print("Customer Data Overview:")
print(df_customers.head())
print(f"\nShape: {df_customers.shape}")
print(f"\nHigh value customers: {df_customers['high_value'].sum()} ({df_customers['high_value'].mean()*100:.1f}%)")


In [ ]:
# Prepare features for Random Forest
# Encode categorical variables
le_city = LabelEncoder()
le_subscription = LabelEncoder()

df_customers['city_encoded'] = le_city.fit_transform(df_customers['city'])
df_customers['subscription_encoded'] = le_subscription.fit_transform(df_customers['subscription_type'])

# Select features
feature_cols = ['age', 'income', 'years_member', 'purchases_last_year', 'city_encoded', 'subscription_encoded']
X = df_customers[feature_cols]
y_regression = df_customers['purchase_amount']
y_classification = df_customers['high_value']

# Split data
X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_regression, test_size=0.2, random_state=42)
_, _, y_class_train, y_class_test = train_test_split(X, y_classification, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")


### Random Forest Regression


In [ ]:
# Train Random Forest Regressor
rf_regressor = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_regressor.fit(X_train, y_reg_train)

# Make predictions
y_reg_train_pred = rf_regressor.predict(X_train)
y_reg_test_pred = rf_regressor.predict(X_test)

# Evaluate
train_rmse = np.sqrt(mean_squared_error(y_reg_train, y_reg_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_test_pred))
train_mae = mean_absolute_error(y_reg_train, y_reg_train_pred)
test_mae = mean_absolute_error(y_reg_test, y_reg_test_pred)
train_r2 = r2_score(y_reg_train, y_reg_train_pred)
test_r2 = r2_score(y_reg_test, y_reg_test_pred)

print("Random Forest Regression Performance:")
print(f"\nTraining Set:")
print(f"  RMSE: {train_rmse:.2f}")
print(f"  MAE: {train_mae:.2f}")
print(f"  R²: {train_r2:.4f}")

print(f"\nTest Set:")
print(f"  RMSE: {test_rmse:.2f}")
print(f"  MAE: {test_mae:.2f}")
print(f"  R²: {test_r2:.4f}")


In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_regressor.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance (Regression)')
plt.tight_layout()
plt.show()


### Random Forest Classification


In [ ]:
# Train Random Forest Classifier
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_class_train)

# Make predictions
y_class_train_pred = rf_classifier.predict(X_train)
y_class_test_pred = rf_classifier.predict(X_test)

# Evaluate
train_acc = accuracy_score(y_class_train, y_class_train_pred)
test_acc = accuracy_score(y_class_test, y_class_test_pred)
train_precision = precision_score(y_class_train, y_class_train_pred)
test_precision = precision_score(y_class_test, y_class_test_pred)
train_recall = recall_score(y_class_train, y_class_train_pred)
test_recall = recall_score(y_class_test, y_class_test_pred)
train_f1 = f1_score(y_class_train, y_class_train_pred)
test_f1 = f1_score(y_class_test, y_class_test_pred)

print("Random Forest Classification Performance:")
print(f"\nTraining Set:")
print(f"  Accuracy: {train_acc:.4f}")
print(f"  Precision: {train_precision:.4f}")
print(f"  Recall: {train_recall:.4f}")
print(f"  F1-Score: {train_f1:.4f}")

print(f"\nTest Set:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall: {test_recall:.4f}")
print(f"  F1-Score: {test_f1:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_class_test, y_class_test_pred))


In [ ]:
# Confusion matrix
import seaborn as sns

cm = confusion_matrix(y_class_test, y_class_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Random Forest Confusion Matrix (Test Set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()


In [ ]:
# Save models
import joblib

joblib.dump(rf_regressor, 'rf_regressor.pkl')
joblib.dump(rf_classifier, 'rf_classifier.pkl')
print("Models saved as 'rf_regressor.pkl' and 'rf_classifier.pkl'")

# Example: Load models
# rf_regressor_loaded = joblib.load('rf_regressor.pkl')
# rf_classifier_loaded = joblib.load('rf_classifier.pkl')


## Section 3: XGBoost/AdaBoost Demo {#section3}

We'll use the same customer data to demonstrate XGBoost and AdaBoost, comparing their performance.


In [ ]:
from xgboost import XGBRegressor, XGBClassifier
from sklearn.ensemble import AdaBoostRegressor, AdaBoostClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

# Use the same customer data from Random Forest section
# X_train, X_test, y_reg_train, y_reg_test, y_class_train, y_class_test are already defined


### XGBoost Regression


In [ ]:
# Train XGBoost Regressor
xgb_regressor = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_regressor.fit(X_train, y_reg_train)

# Make predictions
y_reg_train_pred_xgb = xgb_regressor.predict(X_train)
y_reg_test_pred_xgb = xgb_regressor.predict(X_test)

# Evaluate
train_rmse_xgb = np.sqrt(mean_squared_error(y_reg_train, y_reg_train_pred_xgb))
test_rmse_xgb = np.sqrt(mean_squared_error(y_reg_test, y_reg_test_pred_xgb))
train_r2_xgb = r2_score(y_reg_train, y_reg_train_pred_xgb)
test_r2_xgb = r2_score(y_reg_test, y_reg_test_pred_xgb)

print("XGBoost Regression Performance:")
print(f"\nTraining Set:")
print(f"  RMSE: {train_rmse_xgb:.2f}")
print(f"  R²: {train_r2_xgb:.4f}")

print(f"\nTest Set:")
print(f"  RMSE: {test_rmse_xgb:.2f}")
print(f"  R²: {test_r2_xgb:.4f}")


### AdaBoost Regression


In [ ]:
# Train AdaBoost Regressor
adaboost_regressor = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=5),
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

adaboost_regressor.fit(X_train, y_reg_train)

# Make predictions
y_reg_train_pred_ada = adaboost_regressor.predict(X_train)
y_reg_test_pred_ada = adaboost_regressor.predict(X_test)

# Evaluate
train_rmse_ada = np.sqrt(mean_squared_error(y_reg_train, y_reg_train_pred_ada))
test_rmse_ada = np.sqrt(mean_squared_error(y_reg_test, y_reg_test_pred_ada))
train_r2_ada = r2_score(y_reg_train, y_reg_train_pred_ada)
test_r2_ada = r2_score(y_reg_test, y_reg_test_pred_ada)

print("AdaBoost Regression Performance:")
print(f"\nTraining Set:")
print(f"  RMSE: {train_rmse_ada:.2f}")
print(f"  R²: {train_r2_ada:.4f}")

print(f"\nTest Set:")
print(f"  RMSE: {test_rmse_ada:.2f}")
print(f"  R²: {test_r2_ada:.4f}")


In [ ]:
# Compare XGBoost vs AdaBoost Regression
comparison_reg = pd.DataFrame({
    'Model': ['XGBoost', 'AdaBoost', 'Random Forest'],
    'Train RMSE': [train_rmse_xgb, train_rmse_ada, train_rmse],
    'Test RMSE': [test_rmse_xgb, test_rmse_ada, test_rmse],
    'Test R²': [test_r2_xgb, test_r2_ada, test_r2]
})

print("Regression Model Comparison:")
print(comparison_reg)

# Visualize comparison
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
x = np.arange(len(comparison_reg))
width = 0.35
plt.bar(x - width/2, comparison_reg['Train RMSE'], width, label='Train', alpha=0.8)
plt.bar(x + width/2, comparison_reg['Test RMSE'], width, label='Test', alpha=0.8)
plt.xlabel('Model')
plt.ylabel('RMSE')
plt.title('Regression RMSE Comparison')
plt.xticks(x, comparison_reg['Model'])
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(comparison_reg['Model'], comparison_reg['Test R²'], alpha=0.8)
plt.xlabel('Model')
plt.ylabel('R² Score')
plt.title('Regression R² Comparison')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### XGBoost Classification


In [ ]:
# Train XGBoost Classifier
xgb_classifier = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb_classifier.fit(X_train, y_class_train)

# Make predictions
y_class_train_pred_xgb = xgb_classifier.predict(X_train)
y_class_test_pred_xgb = xgb_classifier.predict(X_test)

# Evaluate
test_acc_xgb = accuracy_score(y_class_test, y_class_test_pred_xgb)
test_precision_xgb = precision_score(y_class_test, y_class_test_pred_xgb)
test_recall_xgb = recall_score(y_class_test, y_class_test_pred_xgb)
test_f1_xgb = f1_score(y_class_test, y_class_test_pred_xgb)

print("XGBoost Classification Performance (Test Set):")
print(f"  Accuracy: {test_acc_xgb:.4f}")
print(f"  Precision: {test_precision_xgb:.4f}")
print(f"  Recall: {test_recall_xgb:.4f}")
print(f"  F1-Score: {test_f1_xgb:.4f}")


### AdaBoost Classification


In [ ]:
# Train AdaBoost Classifier
adaboost_classifier = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

adaboost_classifier.fit(X_train, y_class_train)

# Make predictions
y_class_train_pred_ada = adaboost_classifier.predict(X_train)
y_class_test_pred_ada = adaboost_classifier.predict(X_test)

# Evaluate
test_acc_ada = accuracy_score(y_class_test, y_class_test_pred_ada)
test_precision_ada = precision_score(y_class_test, y_class_test_pred_ada)
test_recall_ada = recall_score(y_class_test, y_class_test_pred_ada)
test_f1_ada = f1_score(y_class_test, y_class_test_pred_ada)

print("AdaBoost Classification Performance (Test Set):")
print(f"  Accuracy: {test_acc_ada:.4f}")
print(f"  Precision: {test_precision_ada:.4f}")
print(f"  Recall: {test_recall_ada:.4f}")
print(f"  F1-Score: {test_f1_ada:.4f}")


In [ ]:
# Compare XGBoost vs AdaBoost vs Random Forest Classification
comparison_class = pd.DataFrame({
    'Model': ['XGBoost', 'AdaBoost', 'Random Forest'],
    'Accuracy': [test_acc_xgb, test_acc_ada, test_acc],
    'Precision': [test_precision_xgb, test_precision_ada, test_precision],
    'Recall': [test_recall_xgb, test_recall_ada, test_recall],
    'F1-Score': [test_f1_xgb, test_f1_ada, test_f1]
})

print("Classification Model Comparison:")
print(comparison_class)

# Visualize comparison
plt.figure(figsize=(14, 4))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(comparison_class))
width = 0.25

for i, metric in enumerate(metrics):
    plt.subplot(1, 4, i+1)
    plt.bar(x - width, comparison_class.iloc[0][metric], width, label='XGBoost', alpha=0.8)
    plt.bar(x, comparison_class.iloc[1][metric], width, label='AdaBoost', alpha=0.8)
    plt.bar(x + width, comparison_class.iloc[2][metric], width, label='Random Forest', alpha=0.8)
    plt.ylabel(metric)
    plt.title(f'{metric} Comparison')
    plt.xticks(x, comparison_class['Model'])
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Feature importance comparison
feature_importance_xgb = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_regressor.feature_importances_
}).sort_values('importance', ascending=False)

print("\nXGBoost Feature Importance:")
print(feature_importance_xgb)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_xgb['feature'], feature_importance_xgb['importance'])
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()


In [ ]:
# Save XGBoost and AdaBoost models
joblib.dump(xgb_regressor, 'xgb_regressor.pkl')
joblib.dump(xgb_classifier, 'xgb_classifier.pkl')
joblib.dump(adaboost_regressor, 'adaboost_regressor.pkl')
joblib.dump(adaboost_classifier, 'adaboost_classifier.pkl')

print("Models saved successfully!")


## Section 4: Common Patterns {#section4}

This section demonstrates common patterns used across all models.


### Data Preprocessing Patterns


In [ ]:
# Pattern 1: Handle missing values
# df['column'].fillna(df['column'].mean(), inplace=True)  # For numeric
# df['column'].fillna(df['column'].mode()[0], inplace=True)  # For categorical

# Pattern 2: Encode categorical variables
# from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# le = LabelEncoder()
# df['encoded'] = le.fit_transform(df['categorical'])

# Pattern 3: Scale features (important for some models)
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

print("Common preprocessing patterns shown above (commented)")


### Model Saving/Loading Patterns


In [ ]:
# Pattern 1: Save/Load TensorFlow/Keras models
# model.save('model.h5')
# loaded_model = keras.models.load_model('model.h5')

# Pattern 2: Save/Load sklearn/XGBoost models
# import joblib
# joblib.dump(model, 'model.pkl')
# loaded_model = joblib.load('model.pkl')

# Pattern 3: Save model predictions
# predictions_df = pd.DataFrame({
#     'actual': y_test,
#     'predicted': y_pred
# })
# predictions_df.to_csv('predictions.csv', index=False)

print("Model saving/loading patterns shown above (commented)")


### Prediction Patterns


In [ ]:
# Pattern 1: Make predictions
# y_pred = model.predict(X_test)

# Pattern 2: Get prediction probabilities (for classification)
# y_proba = model.predict_proba(X_test)

# Pattern 3: Make predictions on new data (ensure same preprocessing!)
# X_new_scaled = scaler.transform(X_new)
# y_new_pred = model.predict(X_new_scaled)

# Pattern 4: Batch predictions for large datasets
# batch_size = 1000
# predictions = []
# for i in range(0, len(X), batch_size):
#     batch = X[i:i+batch_size]
#     pred = model.predict(batch)
#     predictions.extend(pred)

print("Prediction patterns shown above (commented)")


### Evaluation Patterns


In [ ]:
# Pattern 1: Calculate multiple metrics
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# rmse = np.sqrt(mean_squared_error(y_true, y_pred))
# mae = mean_absolute_error(y_true, y_pred)
# r2 = r2_score(y_true, y_pred)

# Pattern 2: Compare train vs test performance
# train_score = model.score(X_train, y_train)
# test_score = model.score(X_test, y_test)

# Pattern 3: Visualize predictions
# plt.scatter(y_true, y_pred)
# plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')

# Pattern 4: Cross-validation evaluation
# from sklearn.model_selection import cross_val_score
# scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print("Evaluation patterns shown above (commented)")


### Summary

In this notebook, you learned:

✅ **LSTM**: Time-series forecasting with sequences, model architecture, training, and evaluation  
✅ **Random Forest**: Regression and classification with feature importance analysis  
✅ **XGBoost/AdaBoost**: Boosting algorithms for regression and classification, performance comparison  
✅ **Common Patterns**: Data preprocessing, model saving/loading, predictions, and evaluation

**Key Takeaways:**
- LSTM requires sequence preparation and works well for time-series data
- Random Forest provides feature importance and handles mixed data types well
- XGBoost often performs better than AdaBoost but requires more tuning
- All models follow similar patterns: preprocessing → training → prediction → evaluation

**Next Steps**: Apply these patterns to the heat risk prediction project!
